In [21]:
from prompt_enhancer import build_prompt as thesis_build_prompt

In [22]:
import json


orig_humaneval_py_path = './humaneval_python_extended.json'

py_problems = json.load(open(orig_humaneval_py_path, 'r'))
print(f'#py_problems={len(py_problems)}')

#py_problems=4


In [23]:
import json

save_orig_humaneval_py_generations_path = './humaneval_python_extended_generations.json'
save_orig_humaneval_py_generations_nodocstrings_path = './humaneval_python_extended_generations_nodocstrings.json'

# save prompt + canonical_solution of each problem in save_orig_humaneval_py_generations_path

py_generations = []

for problem in py_problems:
    py_generations.append(f'{problem["prompt"]}{problem["canonical_solution"]}')
    
with open(save_orig_humaneval_py_generations_path, 'w') as f:
    json.dump(py_generations, f, indent=2)
    print(f'saved #{len(py_generations)} py_generations to {save_orig_humaneval_py_generations_path}')
    
py_generations_nodocstrings = []

for problem in py_problems:
    py_generations_nodocstrings.append(f'{problem["declaration"]}{problem["canonical_solution"]}')
    
with open(save_orig_humaneval_py_generations_nodocstrings_path, 'w') as f:
    json.dump(py_generations_nodocstrings, f, indent=2)
    print(f'saved #{len(py_generations_nodocstrings)} py_generations_nodocstrings to {save_orig_humaneval_py_generations_nodocstrings_path}')

saved #4 py_generations to ./humaneval_python_extended_generations.json
saved #4 py_generations_nodocstrings to ./humaneval_python_extended_generations_nodocstrings.json


In [24]:
orig_humaneval_java_path = './humaneval_java_extended.json'

java_problems = json.load(open(orig_humaneval_java_path, 'r'))
print(f'#java_problems={len(java_problems)}')

#java_problems=4


In [25]:
import os

def apply_rules(build_prompt_func, prompt_version, base_dir='./'):
    base_dir = base_dir or './'
    humaneval_py_java_prompts = [
        build_prompt_func(py_gen_nodocstring, java_problem['declaration'])
        for py_gen_nodocstring, java_problem in zip(py_generations_nodocstrings, java_problems)
    ]
    
    save_prompts_path = os.path.join(base_dir, f'us/humaneval_python_extended_java_prompts_{prompt_version}.json')
    
    with open(save_prompts_path, 'w') as f:
        json.dump(humaneval_py_java_prompts, f, indent=2)
        print(f'saved #{len(humaneval_py_java_prompts)} humaneval_python_extended_java_prompts to {save_prompts_path}')

In [26]:
prompt_template = f"""# TASK
You will be provided with a Python code in triple backticks. Translate the provided Python code to Java.
# INPUT
```
<python-code>
```

---
# OUTPUT
Translate the provided Python code to Java starting with:
```
// language: Java
<java-declaration>"""


def build_prompt(py_generation_nodocstrings, java_declaration):
    return thesis_build_prompt(prompt_template, py_generation_nodocstrings, java_declaration)


def build_prompt_for_gpt(py_generation_nodocstrings, java_declaration):
    return thesis_build_prompt(f'no yapping! just code!\n{prompt_template}', py_generation_nodocstrings, java_declaration)


prompt_version = 'vRULE-COMB-017'  # 'v2006'
prompt_version_gpt = f'vRULE-COMB-017-gpt'  # RULE-COMB-017

apply_rules(build_prompt, prompt_version)
# apply_rules(build_prompt_for_gpt, prompt_version_gpt)

saved #4 humaneval_python_extended_java_prompts to ./us/humaneval_python_extended_java_prompts_vRULE-COMB-017.json


In [27]:
prompt_template = f"""# TASK
You will be provided with a Python code in triple backticks. Translate the provided Python code to Java. Use the exact provided Java declaration, class structure, and imports:
<java-declaration>

# INPUT
Python:
```
# language: Python
<python-code>
```

---
# OUTPUT
Translate the provided Python code to Java starting with:
```
// language: Java
<java-declaration>"""

def build_prompt(py_generation_nodocstrings, java_declaration):
    return thesis_build_prompt(prompt_template, py_generation_nodocstrings, java_declaration)


def build_prompt_for_gpt(py_generation_nodocstrings, java_declaration):
    return thesis_build_prompt(f'no yapping! just code!\n{prompt_template}', py_generation_nodocstrings, java_declaration)


prompt_version = 'vRULE-COMB-018'  # v2003a
prompt_version_gpt = f'vRULE-COMB-018-gpt'  # RULE-COMB-018

apply_rules(build_prompt, prompt_version)
# apply_rules(build_prompt_for_gpt, prompt_version_gpt)

saved #4 humaneval_python_extended_java_prompts to ./us/humaneval_python_extended_java_prompts_vRULE-COMB-018.json


In [28]:
prompt_template = f"""# TASK
You will be provided with a Python code surrounded by triple backticks in the `INPUT` section. Translate the provided Python code to Java, following a step-by-step process to ensure accuracy. Ensure the function logic matches the Python implementation while adhering to Java syntax and conventions. Use the exact provided Java declaration, class structure, and imports:
<java-declaration>

**Steps to follow:**

1. **Understand the Purpose of the Function**: Describe the intended purpose of the function, including the role of each parameter and the expected output type.

2. **Identify Python Constructs Requiring Java Equivalents**:
    - Map each parameter's Python type to its closest Java equivalent.
    - Identify any Python-specific constructs (e.g., `enumerate`, list comprehensions, `lambda` functions) and consider how to implement similar functionality in Java.
    - Identify standard Python functions (e.g., `abs`, `len`, `max`) and find their Java counterparts (e.g., `Math.abs`, `.size()` on collections, `Collections.max`).

3. **Translate Logic and Control Flow**:
    - Convert Python control structures (loops, conditionals) to Java, maintaining the original function's logical flow.
    - Adapt any indexing and conditional checks as needed to follow Java syntax.
    - Handle Python features such as dynamic typing and implicit returns by translating them into Java's more explicit type and structure requirements.

4. **Implement the Java Code**:
    - Construct the Java code following **each component of the original Python function**, ensuring strict adherence to the provided Java declaration and class structure.
    - Ensure that the function logic, including loops, conditions, and variable handling, matches the Python code's intended behavior.

5. **Review and Confirm**:
    - Verify that the Java translation adheres to the provided declaration and maintains the original function's logic and expected output.
    - Ensure all variables, method calls, and syntax conform to Java conventions, and handle any Python-specific behaviors in a way that makes sense in Java.
    - Retain the provided Java class structure, imports, and function name.

# INPUT
```
# language: Python
<python-code>
```

# OUTPUT
Translate the provided Python code to Java starting with:
```
// language: Java
<java-declaration>
"""

# v3: Zero-shot-CoT + clear instr + repeat instr + clear syntax + trained py + trained java
def build_prompt(py_generation_nodocstrings, java_declaration):
    return thesis_build_prompt(prompt_template, py_generation_nodocstrings, java_declaration)


def build_prompt_for_gpt(py_generation_nodocstrings, java_declaration):
    return thesis_build_prompt(f'no yapping! just code!\n{prompt_template}', py_generation_nodocstrings, java_declaration)


prompt_version = 'vRULE-COMB-019' # 'v3001f'
# prompt_version_gpt = f'vRULE-COMB-019-gpt'  # RULE-COMB-019

apply_rules(build_prompt, prompt_version)
# apply_rules(build_prompt_for_gpt, prompt_version_gpt)

saved #4 humaneval_python_extended_java_prompts to ./us/humaneval_python_extended_java_prompts_vRULE-COMB-019.json
